# RLSF — checkpoint selection and val inference


---
## 1 — Host, working tree, disk

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

PY = sys.executable
ROOT = Path.cwd()
print('kernel', PY)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

---
## 2 — Environment and run parameters

In [ ]:
# COMET-Kiwi is a reported column in the selection table, not part of the rule: `ranked()` bands
# on chrF and orders on held-out register distance (src/rlsf/select.py:159). With
# `rlsf.reward.kiwi.gpus: 0` it would also score every checkpoint's dev generations on CPU.
SKIP_KIWI = True

# The full dev slice, 499 segments. A cap here is a deviation from the documented selection path
# and belongs in the pre-registration if it is used.
DEV_LIMIT = 0

# Hours booked on this box, for the section 4 projection.
BUDGET_H = 8.0

KIWI_FLAG = '--skip_kiwi' if SKIP_KIWI else ''
print(f'skip_kiwi={SKIP_KIWI}  dev_limit={DEV_LIMIT}  budget={BUDGET_H} h')

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

In [ ]:
COMET_PY = '.venv-comet/bin/python'

# Only needed if the selection table is to carry the Kiwi column. COMET's pins downgrade
# transformers and numpy, so it gets its own interpreter and never the kernel's.
if not SKIP_KIWI and not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
if not SKIP_KIWI:
    subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)
else:
    print('no COMET worker this session')

In [ ]:
import getpass
import logging
import os

# HF_TOKEN only: the model repo is private and the base model is public. No rater key is read
# in this notebook, because nothing here can spend.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; Phase B makes no paid call'
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

---
## 3 — Adapters

In [ ]:
!hf download prnamhr/style-aware-mt-models --repo-type model --local-dir models

In [ ]:
import json

from src.rlsf.config import load_config
from src.rlsf.select import checkpoints
from src.rlsf.train import arm_path, sidecar

CONFIG = 'configs/rlsf.yaml'
cfg = load_config(CONFIG, require_caps=False)

# The three arms, in the order the pre-registration reports them.
ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}

FOUND = {}
for name, cell in ARMS.items():
    FOUND[cell] = checkpoints(arm_path(cfg['output']['adapter_dir'], cell))
    tags = [t for t, _ in FOUND[cell]]
    print(f"{name:16s} {cell:8s} {len(tags):3d} adapters: {', '.join(tags[:3])} ... {tags[-1]}")

counts = {c: len(v) for c, v in FOUND.items()}
# The arms are selected over the same ladder or the selections are not comparable.
assert len(set(counts.values())) == 1, counts
assert all(v[-1][0] == 'final' for v in FOUND.values()), 'an arm has no final adapter'

In [ ]:
# What came down must be what was trained: the manifests are committed, the weights are not.
for name, cell in ARMS.items():
    log = arm_path(cfg['output']['step_log'], cell)
    man = json.loads(sidecar(log, 'manifest.json').read_text())
    out, arm_dir = man['outcome'], arm_path(cfg['output']['adapter_dir'], cell)
    assert out['adapter_dir'] == str(arm_dir), out['adapter_dir']
    assert out['rollouts'] == 300, out['rollouts']
    print(f"{name:16s} {out['rollouts']} rollouts, omega {man['omega']}, "
          f"adapter {out['adapter_delta']['rel']:.3e} from init, halted={out['stop_reason']}")

---
## 4 — The gate

In [ ]:
N_DEV = sum(1 for line in open(cfg['data']['dev_file']) if line.strip())
N_VAL = sum(1 for line in open('data/splits/val.jsonl') if line.strip())

n_loads = sum(len(v) for v in FOUND.values()) + len(ARMS)
n_select = sum(len(v) for v in FOUND.values()) * (DEV_LIMIT or N_DEV)
n_val = len(ARMS) * N_VAL

print(f"selection  {sum(len(v) for v in FOUND.values())} checkpoints x {DEV_LIMIT or N_DEV} dev "
      f"segments = {n_select:,} generations")
print(f"inference  {len(ARMS)} arms x {N_VAL} val segments = {n_val:,} generations")
print(f"total      {n_select + n_val:,} greedy generations, {n_loads} model loads")

In [ ]:
import time

from src.infer.run import build_zeroshot_user, make_client

# Timed outside outputs/rlsf/select_*/ on purpose: a short file written there would be skipped
# by the real pass (select.py only regenerates under --overwrite) and freeze a truncated
# checkpoint into the selection.
PROBE_N = 8
style = Path(cfg['prompt']['style_instruction_file']).read_text(encoding='utf-8')
dev = [json.loads(x) for x in open(cfg['data']['dev_file']) if x.strip()][:PROBE_N]
val = [json.loads(x) for x in open('data/splits/val.jsonl') if x.strip()][:PROBE_N]

# max_tokens 1024 is the val setting; selection runs at the arm's 192, so the dev rate this
# yields is an over-estimate and the projection errs long.
probe_gen = {**cfg['generator'], 'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 1024,
             'seed': cfg['rlsf']['seed'], 'adapter_path': str(dict(FOUND['w3_0.0'])['final'])}

t0 = time.perf_counter()
probe = make_client(probe_gen)
load_s = time.perf_counter() - t0


def rate(client, rows):
    t = time.perf_counter()
    for row in rows:
        client.complete(style, build_zeroshot_user(row['input']))
    return (time.perf_counter() - t) / len(rows)


dev_s, val_s = rate(probe, dev), rate(probe, val)
print(f'{load_s:.0f}s model load')
print(f'{dev_s:.2f}s per dev segment, {val_s:.2f}s per val segment')

In [ ]:
select_h = (n_select * dev_s + (n_loads - len(ARMS)) * load_s) / 3600
infer_h = (n_val * val_s + len(ARMS) * load_s) / 3600
print(f'selection  {select_h:5.1f} h')
print(f'inference  {infer_h:5.1f} h')
print(f'total      {select_h + infer_h:5.1f} h against {BUDGET_H:.1f} h booked')

if select_h + infer_h > 0.8 * BUDGET_H:
    print('\nThis does not fit with room to spare. Pull a lever below before section 5.')
else:
    print('\nFits. Section 5 may start.')

In [ ]:
import gc

import torch

# The probe's weights are dead once the projection is read; the selection pass loads its own.
del probe
gc.collect()
torch.cuda.empty_cache()
print(f'{torch.cuda.memory_reserved() / 2**30:.2f} GiB still reserved')

---
## 5 — Selection, one arm per cell

In [ ]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_0.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

In [ ]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_2.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

In [ ]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_6.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

---
## 6 — Freeze the selection into the eval configs

Done here rather than by hand. A hand-edited `adapter_path` under time pressure on a rented box
is how an arm ends up scored on another arm's checkpoint, and the val hypotheses carry no record
of which adapter wrote them beyond this line.

In [ ]:
import re

SELECTED = {}
for cell in ARMS.values():
    sel = json.loads(Path(f'results/rlsf_select_{cell}.json').read_text())
    path = Path(sel['selected_path'])
    arm_dir = arm_path(cfg['output']['adapter_dir'], cell)
    assert (path / 'adapter_config.json').exists(), path
    # The selected checkpoint must belong to this arm, not the one selected before it.
    assert path == arm_dir or arm_dir in path.parents, (cell, path)

    cfg_path = Path(f'configs/rlsf_eval_{cell}.yaml')
    text = cfg_path.read_text(encoding='utf-8')
    new, n = re.subn(
        r'^(\s*adapter_path:\s*)\S+.*$',
        rf'\g<1>{path}    # {sel["selected"]}, selected on the dev slice',
        text, count=1, flags=re.M,
    )
    assert n == 1, f'no adapter_path line in {cfg_path}'
    cfg_path.write_text(new, encoding='utf-8')
    SELECTED[cell] = sel
    print(f"{cell:8s} {sel['selected']:8s} chrF {sel['rows'][0]['chrF']:6.2f}  ->  {cfg_path}")

In [ ]:
# The rule's own ordering, per arm: the chrF band first, then held-out register distance.
for cell, sel in SELECTED.items():
    by_tag = {r['tag']: r for r in sel['rows']}
    print(f"\n{cell}  ({sel['n_segments']} dev segments, margin {sel['adequacy_margin']})")
    print(f"  {'tag':8s} {'chrF':>7s} {'BLEU':>7s} {'dist_heldout':>13s} {'dist_reward':>12s}")
    for tag in sel['ranked'][:5]:
        r = by_tag[tag]
        mark = ' <-' if tag == sel['selected'] else ''
        print(f"  {tag:8s} {r['chrF']:7.2f} {r['BLEU']:7.2f} {r['dist_heldout']:13.4f} "
              f"{r['dist_reward']:12.4f}{mark}")

In [ ]:
!git diff -- configs/

---
## 7 — Val inference

In [ ]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_0.0.yaml

In [ ]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_2.0.yaml

In [ ]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_6.0.yaml

---
## 8 — Verify before teardown

In [ ]:
import yaml

VAL = [json.loads(x) for x in open('data/splits/val.jsonl') if x.strip()]
HYPS = {}
for cell in ARMS.values():
    name = yaml.safe_load(Path(f'configs/rlsf_eval_{cell}.yaml').read_text())['output']['name']
    path = Path('outputs') / f'{name}_val.jsonl'
    rows = [json.loads(x) for x in open(path) if x.strip()]
    HYPS[cell] = (path, rows)

    assert len(rows) == len(VAL), f'{path}: {len(rows)} rows, expected {len(VAL)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, VAL)), f'{path}: source misalignment'
    errors = [r for r in rows if r.get('error')]
    empty = [r for r in rows if not r['prediction'].strip()]
    assert not errors, f'{path}: {len(errors)} segments recorded an error'
    assert not empty, f'{path}: {len(empty)} empty predictions'
    assert {r['condition'] for r in rows} == {'peft'}
    print(f'{path}  {len(rows)} rows, no errors, no empties')

In [ ]:
from sacrebleu.metrics import CHRF

from src.eval.quick import _marker_rate

# Not the result -- Phase C is. A catastrophically broken arm is cheaper to find now than after
# the box is destroyed.
chrf = CHRF()
for cell, (path, rows) in HYPS.items():
    preds = [r['prediction'] for r in rows]
    refs = [r['output'] for r in rows]
    print(f"{cell:8s} chrF {chrf.corpus_score(preds, [refs]).score:6.2f}  "
          f"marker_rate {_marker_rate(preds):5.2f}  "
          f"adapter {SELECTED[cell]['selected_path']}")

In [ ]:
# The seal: nothing in this session may have read the test split.
for cell, (path, _) in HYPS.items():
    assert 'test' not in path.name, path
    u = json.loads(path.with_name(f'{path.stem}_usage.json').read_text())
    assert u['cost_usd'] == 0.0, u
    print(f"{cell:8s} {u['calls']} local calls, ${u['cost_usd']:.2f}")
print('\n0 paid calls, $0.00 spent in Phase B')

---
## 9 — Commit, push, tear down

In [ ]:
# models/ is gitignored, so nothing model-sized travels. The dev generations are the evidence
# the selection table summarizes; they are the one judgement call in this list.
tables = sorted(str(p) for p in Path('results').glob('rlsf_select_*.json'))
hyps = sorted(str(p) for cell in ARMS.values() for p in Path('outputs').glob(f'rlsf_{cell}_val*'))
devgen = sorted(str(p) for p in Path('outputs/rlsf').glob('select_*'))
configs = [f'configs/rlsf_eval_{cell}.yaml' for cell in ARMS.values()]

mib = sum(f.stat().st_size for d in devgen for f in Path(d).rglob('*')) / 2**20
print(f'dev generations: {mib:.0f} MiB across {len(devgen)} directories\n')
print('git add ' + ' '.join(tables + configs + hyps + devgen))
print('git commit -m "feat: rlsf checkpoint selection and val hypotheses for the three arms"')
print('git push')

In [ ]:
# Run after pushing. Teardown is safe only once the remote has the commit.
!git status --short
!git fetch --quiet origin
!git log --oneline -1 origin/$(git rev-parse --abbrev-ref HEAD)